# Label-free self-supervised stereo depth

Re-implementation of the stereo architecture from *A Learned Stereo Depth System for
Robotic Manipulation in Homes* (Shankar et al., arXiv:2109.11644), trained **without any
ground-truth disparity or depth**.

### The one rule

| Mode | Ground truth |
|---|---|
| `train_unlabeled` | **never read** |
| `adapt_unlabeled` | **never read** |
| `evaluate` | read, for metrics only, on a frozen checkpoint |

Every ground-truth cell is guarded by `if MODE == "evaluate"` and refuses to run otherwise.

### How to use this notebook

1. **Edit the Control Panel (cell 3) — and nothing else.** Every knob lives there; the rest
   of the notebook derives from it. Adding a dataset, changing resolution or switching to
   evaluation are all one-line edits in that single cell.
2. **Run all.** The preflight cell right after it prints exactly what will happen and warns
   about anything inconsistent *before* any training starts.
3. Settings -> Accelerator -> **GPU T4 x2** or **P100**, and Internet **on** if you want the
   notebook to download datasets itself.

## 1. Configuration guide

What every knob in the Control Panel means. **You do not need to read this to start** —
the defaults train on Middlebury and evaluate with the paper's Middlebury metric set.

### Where the code comes from
| Setting | Meaning |
|---|---|
| `GITHUB_USER` | Your GitHub username. Leave `""` to load the code from an attached Kaggle Dataset instead (no GitHub account needed). |
| `GITHUB_REPO` | Repository name, if you pushed one. |

### What to run
| Setting | Meaning |
|---|---|
| `MODE` | `train_unlabeled` = Stages 1+2 from random init. `adapt_unlabeled` = Stage 3, fine-tune an existing checkpoint on your own camera. `evaluate` = Stage 4, freeze a checkpoint and measure it against ground truth. |

### Which data
| Setting | Meaning |
|---|---|
| `DATASETS` | `name -> sampling weight`. The weight is the **share of training draws, not a share of images**: a 15-scene set at 0.5 is drawn as often as a 20 000-frame set at 0.5. Set a weight to `0` to exclude it — only non-zero entries are downloaded. This one dict drives downloading, previewing and training. |
| `ATTACHED` | `name -> /kaggle/input/...` for datasets you **attach** instead of downloading. Anything listed here is read straight from that path, so Kaggle's 20 GB working-directory quota never applies. **This is how to use Scene Flow on Kaggle.** The mirror's internal folder layout does not matter — see below. |
| `CUSTOM_DATA` | Your own capture for `adapt_unlabeled`: a folder with `left/` and `right/` and no labels. |
| `DATASET_ROOT` | Where *downloaded* datasets go. Ignored for anything in `ATTACHED`. |

#### Using Scene Flow (or any large dataset) without downloading it

Scene Flow is 132 GB officially — far beyond Kaggle's quota. Attach a community mirror instead:

1. **Add Data** → search `sceneflow` → attach it.
2. Note the path it appears at under `/kaggle/input/`.
3. In the Control Panel:

```python
DATASETS = {"sceneflow": 1.0}                       # give it a weight
ATTACHED = {"sceneflow": "/kaggle/input/sceneflow"} # and point at it
```

Nothing is downloaded for it. **The mirror's internal layout does not matter** — the loader
searches the attached directory (up to 5 levels deep) for either structure:

| Layout | Looks like |
|---|---|
| official | `frames_finalpass/TRAIN/<A\|B\|C>/<scene>/left/*.png` + `disparity/TRAIN/...` |
| subset | `train/image_clean/left/*.png` + `train/disparity/left/*.pfm` |

It falls back to `frames_cleanpass` if there is no `frames_finalpass`, and maps `TEST` onto
the subset release's `val` directory. The "Attached datasets" cell prints exactly what it
resolved, and if a mirror is not recognised it prints that mirror's real directory tree
rather than failing opaquely.

A mirror with **images but no disparity is still fully usable for training** — training here
is label-free. Disparity is only needed for Stage 4 benchmarking, and the loader says so
explicitly if you try to benchmark against an images-only copy.

### Resolution and disparity range
| Setting | Meaning |
|---|---|
| `TRAIN_WIDTH`, `TRAIN_HEIGHT` | Every sample is **resized** to this (never cropped), so batches are rectangular. Width also sets the disparity search range: `num_disparities = min(width // 2, 384)`. |
| `DOWNSAMPLE` | Cost volume resolution: `4` = 1/4 (the paper's low-res variant), `8` = 1/8 (high-res variant, cheaper). |
| `MAX_DISPARITY_CAP` | The `384` in the rule above. Raise to `512` for full/half-resolution Middlebury, whose disparities exceed 384 px — the paper uses 512 there. |

**The number that matters is your disparity range, not your image size.** `d_max = f·B/Z_min`
at your native width, scaled by `TRAIN_WIDTH / NATIVE_WIDTH`. The preflight cell checks this
against your data and warns if the model cannot reach far enough.

### Optimisation
| Setting | Meaning |
|---|---|
| `EPOCHS`, `BATCH_SIZE`, `LEARNING_RATE` | Standard. Batch size is bounded by the cost volume, which grows with `TRAIN_WIDTH × TRAIN_HEIGHT × num_disparities / DOWNSAMPLE³` — the preflight prints the per-image cost. |
| `NUM_WORKERS` | Dataloader processes. `2` suits Kaggle. |

### Loss weights — the label-free objective
`L = W_PHOTOMETRIC·L_photo + W_SMOOTHNESS·L_smooth + W_LEFT_RIGHT·L_lr + W_LOW_RESOLUTION·(photo+smooth at cost-volume scale) + W_PSEUDO·ramp·L_pseudo + W_CONFIDENCE·L_conf`

| Setting | Meaning |
|---|---|
| `W_PHOTOMETRIC` | SSIM+L1 reconstruction — the primary signal. Leave at 1.0 and scale the others relative to it. |
| `W_SMOOTHNESS` | Edge-aware smoothness. Too high collapses disparity toward a constant; too low leaves textureless regions noisy. |
| `W_LEFT_RIGHT` | Geometric agreement between the two disparity maps. Also what handles occlusions. |
| `W_LOW_RESOLUTION` | Applies photometric+smoothness to the soft-argmin output too. Without it the cost volume gets gradient only through the refinement net, which learns to ignore a bad coarse input rather than fix it. |
| `W_PSEUDO` | Teacher pseudo-label consistency (Stage 2). |
| `W_CONFIDENCE` | Label-free matchability target. **This one is my addition, not the paper's, and is unvalidated** — set to `0.0` to disable it while keeping the architecture intact. |

### Teacher (Stage 2 self-training)
| Setting | Meaning |
|---|---|
| `TEACHER_START_EPOCH` | Pseudo-labelling is off before this. The model needs to be roughly right first, or it teaches itself its own mistakes. |
| `TEACHER_EMA_DECAY` | `θ_teacher ← m·θ_teacher + (1−m)·θ_student`. Higher = slower, more stable teacher. |
| `TEACHER_RAMP_EPOCHS` | Linear ramp-in of `W_PSEUDO`, so the objective does not step-change when the teacher starts. |
| `FILTER_CONFIDENCE`, `FILTER_LR_PIXELS`, `FILTER_PHOTOMETRIC` | A teacher pixel becomes a pseudo-label only if it passes **all** of these. Watch `pseudo_cov` in the training log: near 0 means the filter rejects everything, near 1 means it copies the teacher's errors wholesale. |

### Evaluation (Stage 4)
| Setting | Meaning |
|---|---|
| `PROTOCOL` | Which benchmark protocol. `EVAL_ROOT` is **derived from this** — no second path to keep in sync. |
| `CHECKPOINT` | `None` = use `best.pt` from this notebook's training run. |
| `POSTPROCESS_CONFIDENCE`, `POSTPROCESS_MIN_REGION` | The paper's on-robot filter (`exp(matchability) ≥ 0.25`, region ≥ 2000 px). The paper's **tables use raw output**, so this is always reported as a separate row. |

---

### Common recipes

<details><summary><b>Train on Middlebury only (the default)</b></summary>

```python
MODE = "train_unlabeled"
DATASETS = {"middlebury": 1.0}
```
</details>

<details><summary><b>Add KITTI to the training mixture</b></summary>

```python
DATASETS = {"middlebury": 0.5, "kitti2015": 0.3, "kitti2012": 0.2}
```
One edit. Downloading, previewing and the training mixture all follow automatically.
</details>

<details><summary><b>Use Scene Flow from an attached Kaggle dataset (not downloaded)</b></summary>

```python
DATASETS = {"sceneflow": 1.0}
ATTACHED = {"sceneflow": "/kaggle/input/sceneflow"}   # whatever path Add Data gave you
PROTOCOL = "sceneflow"                                # for MODE="evaluate"
```
Two lines. Nothing is downloaded, and the mirror's internal folder layout is discovered
automatically.
</details>

<details><summary><b>Train on your own unlabeled camera</b></summary>

```python
MODE        = "adapt_unlabeled"
CUSTOM_DATA = "/kaggle/input/my-stereo-camera"   # contains left/ and right/
INIT_CHECKPOINT = "/kaggle/working/outputs/train_unlabeled/best.pt"
```
Adaptation starts from an existing checkpoint at a reduced learning rate (`ADAPT`).
</details>

<details><summary><b>Small images, e.g. 224x224</b></summary>

```python
TRAIN_HEIGHT = 224
TRAIN_WIDTH  = 224      # -> num_disparities = 112, max disparity 107 px
BATCH_SIZE   = 24       # 6 MB/image instead of 79 at 640x384
```
Do **not** upscale small images to 640x384: it multiplies disparity by 2.9, adds no
information, and costs 13x the cost-volume memory. And do not squash a wide capture into a
square — horizontal resolution *is* depth precision (`δZ = Z²/(fB)·δd`, and halving the
width halves `f`).
</details>

<details><summary><b>Evaluate a checkpoint against ground truth</b></summary>

```python
MODE       = "evaluate"
PROTOCOL   = "middlebury2014"
CHECKPOINT = "/kaggle/working/outputs/train_unlabeled/best.pt"
```
</details>

## 2. Control Panel

**This is the only cell you need to edit.** Everything below derives from it.

In [ ]:
# =============================================================================
#  CONTROL PANEL  --  the only cell you normally edit
# =============================================================================

# --- Where the code comes from -----------------------------------------------
# Leave GITHUB_USER empty ("") to load the code from an attached Kaggle Dataset
# instead. That needs no GitHub account and works with Internet off:
#   Add Data -> Upload -> select your local stereo-depth/ folder.
GITHUB_USER = ""                       # e.g. "your-github-name"
GITHUB_REPO = "stereo-depth"

# --- What to run -------------------------------------------------------------
# "train_unlabeled"  Stages 1+2, from random init, on benchmark images (no labels)
# "adapt_unlabeled"  Stage 3, fine-tune an existing checkpoint on your own camera
# "evaluate"         Stage 4, freeze a checkpoint and measure it against ground truth
MODE = "train_unlabeled"

# --- Which data --------------------------------------------------------------
# name -> sampling weight. The weight is the share of training DRAWS, not of images:
# a 15-scene set at 0.5 is drawn as often as a 20000-frame set at 0.5.
# Set a weight to 0 to exclude a dataset -- only non-zero entries are downloaded.
# This single dict drives downloading, previewing and the training mixture.
DATASETS = {
    "middlebury": 1.0,     #  155 MB   15 indoor scenes, high resolution
    "kitti2015":  0.0,     #  1.7 GB   200 outdoor driving pairs
    "kitti2012":  0.0,     #  2.0 GB   194 outdoor driving pairs
    "eth3d":      0.0,     #  1.1 GB   needs 7z: !apt-get install -y -qq p7zip-full
    "sceneflow":  0.0,     #  132 GB   DO NOT download on Kaggle -- attach a mirror and
                           #           point ATTACHED at it (see just below)
}

# Datasets you have ATTACHED as Kaggle inputs instead of downloading.
# Any name listed here is used straight from that path -- nothing is downloaded
# for it, and Kaggle's 20 GB working-directory quota never comes into play.
# This is how to use Scene Flow on Kaggle:
#   Add Data -> search "sceneflow" -> attach, then give its path here and set a
#   non-zero weight in DATASETS above.
# The exact folder layout inside the mirror does not matter: the loader searches
# the attached directory for either the official layout
# (frames_finalpass/TRAIN/...) or the smaller FlyingThings3D_subset layout
# (train/image_clean/left/...), however deeply it is nested.
ATTACHED = {
    # "sceneflow": "/kaggle/input/sceneflow",
}

DATASET_ROOT = "/kaggle/working/datasets"
CUSTOM_DATA  = "/kaggle/input/my-stereo-camera"   # adapt_unlabeled: needs left/ and right/

# --- Resolution and disparity range ------------------------------------------
# Samples are RESIZED to this (never cropped). Width sets the search range:
#   num_disparities = min(TRAIN_WIDTH // 2, MAX_DISPARITY_CAP)
TRAIN_HEIGHT      = 384
TRAIN_WIDTH       = 640      # -> num_disparities = min(320, 384) = 320
DOWNSAMPLE        = 4        # cost volume at 1/4 resolution (8 = cheaper, coarser)
MAX_DISPARITY_CAP = 384      # raise to 512 for full/half-res Middlebury (the paper's value)

# --- Optimisation ------------------------------------------------------------
EPOCHS        = 30
BATCH_SIZE    = 8
LEARNING_RATE = 1e-4
NUM_WORKERS   = 2

# --- Loss weights (the label-free objective) ---------------------------------
W_PHOTOMETRIC    = 1.0    # SSIM+L1 reconstruction -- the primary signal
W_SMOOTHNESS     = 0.1    # edge-aware; too high collapses disparity to a constant
W_LEFT_RIGHT     = 0.5    # geometric agreement; also handles occlusions
W_LOW_RESOLUTION = 0.5    # same terms on the soft-argmin output (feeds the cost volume)
W_PSEUDO         = 1.0    # teacher pseudo-label consistency (Stage 2)
W_CONFIDENCE     = 0.05   # label-free matchability target -- NOT from the paper,
                          # unvalidated; set 0.0 to disable

# --- Teacher / student (Stage 2) ---------------------------------------------
TEACHER_ENABLED     = True
TEACHER_START_EPOCH = 8       # pseudo-labelling is off before this epoch
TEACHER_EMA_DECAY   = 0.999   # higher = slower, more stable teacher
TEACHER_RAMP_EPOCHS = 5       # linear ramp-in of W_PSEUDO
FILTER_CONFIDENCE   = 0.5     # a teacher pixel must pass ALL THREE filters
FILTER_LR_PIXELS    = 1.0     # left-right agreement, in pixels
FILTER_PHOTOMETRIC  = 0.15    # photometric residual

# --- Stage 3 adaptation overrides (used only when MODE == "adapt_unlabeled") --
INIT_CHECKPOINT = None        # None -> best.pt from this notebook's train_unlabeled run
ADAPT = dict(
    lr_scale=0.1,             # adaptation learning rate = LEARNING_RATE * lr_scale
    teacher_start_epoch=2,    # the model is already usable, so no long warm-up
)

# --- Evaluation (Stage 4) ----------------------------------------------------
# EVAL_ROOT is DERIVED from PROTOCOL -- there is no second path to keep in sync.
PROTOCOL   = "middlebury2014"   # middlebury2014 | sceneflow | eth3d | kitti2015 |
                                # kitti2012 | custom_folder
CHECKPOINT = None               # None -> best.pt from this notebook's training run
EVAL_MAX_SAMPLES = None         # None = all images; set a small int for a quick check
POSTPROCESS_CONFIDENCE = 0.25   # the paper's exp(matchability) threshold
POSTPROCESS_MIN_REGION = 2000   # the paper's minimum depth-region size, in pixels

# =============================================================================
print(f"MODE = {MODE}")
print("Run the next cells; the preflight will report exactly what this means.")

## 3. Get the code

Two ways, and the cell picks whichever is available: an attached Kaggle Dataset (no GitHub
account, works offline) or a `git clone` if you set `GITHUB_USER`.

In [ ]:
import glob, os, shutil, subprocess, sys

REPO_DIR = "/kaggle/working/stereo-depth"
REPO_URL = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git" if GITHUB_USER else None


def _looks_like_this_repo(directory: str) -> bool:
    """A directory is the source tree if it has the files this notebook imports --
    not just any dataset that happens to be attached."""
    return (os.path.isfile(os.path.join(directory, "train.py"))
            and os.path.isfile(os.path.join(directory, "stereo", "model", "stereo_net.py")))


def _find_attached_dataset():
    """Search every attached Kaggle input, regardless of the dataset's slug --
    Kaggle names it from whatever title you typed, so a fixed name would only
    work by coincidence. Also looks one level down, since Kaggle often nests an
    uploaded folder inside another folder of the same name."""
    if not os.path.isdir("/kaggle/input"):
        return None
    for candidate in sorted(glob.glob("/kaggle/input/*")) + sorted(glob.glob("/kaggle/input/*/*")):
        if os.path.isdir(candidate) and _looks_like_this_repo(candidate):
            return candidate
    return None


if not os.path.exists(REPO_DIR):
    source = _find_attached_dataset()
    if source is not None:
        print(f"found the repository attached as a Kaggle dataset: {source}")
        shutil.copytree(source, REPO_DIR)
    elif REPO_URL is not None:
        print(f"cloning {REPO_URL}")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    else:
        raise RuntimeError(
            "Could not find the stereo-depth source code.\n\n"
            "GITHUB_USER is empty and no attached Kaggle dataset contains train.py + "
            "stereo/model/stereo_net.py. Pick one of:\n\n"
            "  A) Add Data -> Upload -> select your local stereo-depth/ folder as a new "
            "Kaggle Dataset, attach it to this notebook, then re-run this cell. Needs no "
            "GitHub account and works with Internet off.\n\n"
            "  B) Push the repo to GitHub and set GITHUB_USER in the Control Panel.")

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# Fast sanity check that the checkout is sound (~15 s).
subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q", "--no-header"], check=False)

## 4. Preflight — what your settings actually mean

Resolves every derived value, explains the consequences of your Control Panel choices, and
warns about inconsistencies **before** anything is downloaded or trained.

In [ ]:
from stereo.data.download import RECIPES
from stereo.geometry import compute_num_disparities

# How each dataset name maps to a training dataset spec. You do not edit this --
# it is driven by the DATASETS dict in the Control Panel.
DATASET_TYPES = {
    "middlebury": ("middlebury", {}),
    "eth3d":      ("eth3d",      {}),
    "kitti2015":  ("kitti",      {"version": "2015", "reference_frames_only": False}),
    "kitti2012":  ("kitti",      {"version": "2012", "reference_frames_only": False}),
    "sceneflow":  ("sceneflow",  {"split": "TRAIN"}),
}
PROTOCOL_TO_DATASET = {recipe.protocol: name for name, recipe in RECIPES.items()}

warnings = []

# --- resolve mode ------------------------------------------------------------
assert MODE in ("train_unlabeled", "adapt_unlabeled", "evaluate"), f"bad MODE: {MODE}"
OUTPUT_DIR = f"/kaggle/working/outputs/{MODE}"

# --- resolve datasets --------------------------------------------------------
ACTIVE_DATASETS = {name: weight for name, weight in DATASETS.items() if weight > 0}
for name in DATASETS:
    if name not in DATASET_TYPES:
        warnings.append(f"unknown dataset name {name!r}; known: {sorted(DATASET_TYPES)}")

def dataset_root_for(name):
    """Attached Kaggle input wins over the download location."""
    if name in ATTACHED:
        return ATTACHED[name]
    return os.path.join(DATASET_ROOT, RECIPES[name].usage_root)

# Only non-attached datasets are ever downloaded.
TO_DOWNLOAD = [n for n in ACTIVE_DATASETS if n not in ATTACHED]

# --- resolve checkpoints -----------------------------------------------------
RESOLVED_INIT = INIT_CHECKPOINT or "/kaggle/working/outputs/train_unlabeled/best.pt"
RESOLVED_CHECKPOINT = CHECKPOINT or f"{OUTPUT_DIR}/best.pt"
if MODE == "evaluate" and CHECKPOINT is None:
    # In evaluate mode there is no training run in this notebook to take best.pt from.
    RESOLVED_CHECKPOINT = "/kaggle/working/outputs/train_unlabeled/best.pt"

# --- resolve the evaluation root FROM the protocol (no second path to sync) ---
if PROTOCOL == "custom_folder":
    EVAL_ROOT = CUSTOM_DATA
elif PROTOCOL in PROTOCOL_TO_DATASET:
    EVAL_ROOT = dataset_root_for(PROTOCOL_TO_DATASET[PROTOCOL])
else:
    raise ValueError(f"unknown PROTOCOL {PROTOCOL!r}; known: "
                     f"{sorted(list(PROTOCOL_TO_DATASET) + ['custom_folder'])}")

# --- resolve the disparity range --------------------------------------------
NUM_DISPARITIES = compute_num_disparities(TRAIN_WIDTH, DOWNSAMPLE, MAX_DISPARITY_CAP)
MAX_DISPARITY_PX = (NUM_DISPARITIES // DOWNSAMPLE - 1) * DOWNSAMPLE - 1
cost_volume_mb = 16 * (NUM_DISPARITIES // DOWNSAMPLE) * (TRAIN_HEIGHT // DOWNSAMPLE) \
                 * (TRAIN_WIDTH // DOWNSAMPLE) * 4 / 1e6

# --- checks ------------------------------------------------------------------
if MODE == "train_unlabeled" and not ACTIVE_DATASETS:
    warnings.append("no dataset has a non-zero weight, so there is nothing to train on. "
                    "Set at least one entry in DATASETS above 0.")
if MODE == "adapt_unlabeled":
    if not os.path.isdir(CUSTOM_DATA):
        warnings.append(f"CUSTOM_DATA does not exist: {CUSTOM_DATA}. Attach your capture "
                        "as a Kaggle Dataset (a folder with left/ and right/).")
    if not os.path.isfile(RESOLVED_INIT):
        warnings.append(f"no checkpoint to adapt from at {RESOLVED_INIT}. Run "
                        "MODE='train_unlabeled' first, or set INIT_CHECKPOINT. Adaptation "
                        "from random weights is not what Stage 3 is for.")
if MODE == "evaluate":
    if not os.path.isfile(RESOLVED_CHECKPOINT):
        warnings.append(f"no checkpoint to evaluate at {RESOLVED_CHECKPOINT}. Train one "
                        "first or set CHECKPOINT.")
    needed = PROTOCOL_TO_DATASET.get(PROTOCOL)
    if needed and needed not in ACTIVE_DATASETS:
        warnings.append(f"PROTOCOL={PROTOCOL!r} needs the {needed!r} dataset, but its "
                        f"weight in DATASETS is 0, so it will not be downloaded. "
                        f"Set DATASETS[{needed!r}] above 0.")
if "eth3d" in ACTIVE_DATASETS and not shutil.which("7z"):
    warnings.append("eth3d needs 7z to extract. Run in a cell first: "
                    "!apt-get install -y -qq p7zip-full")
if "sceneflow" in ACTIVE_DATASETS and "sceneflow" not in ATTACHED:
    warnings.append("sceneflow is 132 GB and will exceed Kaggle's 20 GB working-dir quota. "
                    "Attach a Kaggle mirror instead and add it to ATTACHED, e.g. "
                    "ATTACHED = {'sceneflow': '/kaggle/input/sceneflow'}.")
for name, path in ATTACHED.items():
    if name not in DATASET_TYPES:
        warnings.append(f"ATTACHED has unknown dataset name {name!r}; "
                        f"known: {sorted(DATASET_TYPES)}")
    elif not os.path.isdir(path):
        warnings.append(f"ATTACHED[{name!r}] does not exist: {path}. Use Add Data to attach "
                        "it, then check the exact path under /kaggle/input.")
    elif DATASETS.get(name, 0) <= 0 and PROTOCOL_TO_DATASET.get(PROTOCOL) != name:
        warnings.append(f"{name!r} is attached but its weight in DATASETS is 0, so it will "
                        "not be used. Give it a non-zero weight.")
if BATCH_SIZE * cost_volume_mb > 11000:
    warnings.append(f"BATCH_SIZE={BATCH_SIZE} needs ~{BATCH_SIZE * cost_volume_mb / 1000:.1f} GB "
                    "for the cost volume alone, which will likely OOM on a 16 GB T4/P100. "
                    "Reduce BATCH_SIZE or TRAIN_WIDTH.")

# --- report ------------------------------------------------------------------
print("=" * 74)
print(f"MODE          : {MODE}")
print(f"output        : {OUTPUT_DIR}")
if MODE == "train_unlabeled":
    described = ", ".join(f"{n} (weight {w}{', attached' if n in ATTACHED else ''})"
                          for n, w in ACTIVE_DATASETS.items())
    print(f"training on   : {described or 'NOTHING'}")
    print("                images only -- no ground truth is read in this mode")
    if TO_DOWNLOAD:
        print(f"will download : {', '.join(TO_DOWNLOAD)}")
    if ATTACHED:
        print(f"attached      : {', '.join(f'{n} -> {p}' for n, p in ATTACHED.items())}")
elif MODE == "adapt_unlabeled":
    print(f"adapting      : {RESOLVED_INIT}")
    print(f"           on : {CUSTOM_DATA} (images only)")
    print(f"learning rate : {LEARNING_RATE * ADAPT['lr_scale']:.1e} "
          f"(= LEARNING_RATE x {ADAPT['lr_scale']})")
else:
    print(f"evaluating    : {RESOLVED_CHECKPOINT}")
    print(f"protocol      : {PROTOCOL}")
    print(f"eval data     : {EVAL_ROOT}   <- derived from PROTOCOL")
    print("                GROUND TRUTH IS READ HERE, for metrics only")
print("-" * 74)
print(f"resolution    : {TRAIN_WIDTH} x {TRAIN_HEIGHT}  (resized, never cropped)")
print(f"disparities   : {NUM_DISPARITIES} = min({TRAIN_WIDTH} // 2, {MAX_DISPARITY_CAP}), "
      f"floored to a multiple of {DOWNSAMPLE}")
print(f"              : the model can see disparities of 0 .. {MAX_DISPARITY_PX} px")
print(f"              : i.e. objects no closer than  Z_min = f * B / {MAX_DISPARITY_PX}")
print(f"cost volume   : {cost_volume_mb:.0f} MB per image, "
      f"~{BATCH_SIZE * cost_volume_mb / 1000:.1f} GB at BATCH_SIZE={BATCH_SIZE}")
if MODE != "evaluate":
    teacher_start = ADAPT["teacher_start_epoch"] if MODE == "adapt_unlabeled" else TEACHER_START_EPOCH
    print(f"teacher       : {'starts at epoch ' + str(teacher_start) if TEACHER_ENABLED else 'disabled'}"
          f"{f', ramped over {TEACHER_RAMP_EPOCHS} epochs' if TEACHER_ENABLED else ''}")
    print(f"objective     : {W_PHOTOMETRIC}*photo + {W_SMOOTHNESS}*smooth + "
          f"{W_LEFT_RIGHT}*lr + {W_LOW_RESOLUTION}*lowres + {W_PSEUDO}*pseudo + "
          f"{W_CONFIDENCE}*conf")
    print("                no ground-truth term exists in this objective")
print("=" * 74)

if warnings:
    print(f"\n{len(warnings)} WARNING(S):")
    for i, message in enumerate(warnings, 1):
        print(f"  {i}. {message}")
else:
    print("\nNo problems found.")

## 5. Datasets

Downloads exactly the datasets with a non-zero weight in `DATASETS`. Nothing else to edit.

**Training reads images only** — ground truth in these datasets is untouched until Stage 4.

In [ ]:
from stereo.data.download import describe, prepare, verify
print(describe())

In [ ]:
# Downloads whatever has a non-zero weight in DATASETS. In adapt mode your own
# capture is used instead, so nothing is downloaded unless you also want to evaluate.
to_fetch = list(ACTIVE_DATASETS)
if MODE == "evaluate":
    needed = PROTOCOL_TO_DATASET.get(PROTOCOL)
    if needed and needed not in to_fetch:
        to_fetch.append(needed)
if MODE == "adapt_unlabeled":
    to_fetch = [n for n in to_fetch if n == PROTOCOL_TO_DATASET.get(PROTOCOL)]

# Anything in ATTACHED is already on disk as a Kaggle input -- never download it.
skipped = [n for n in to_fetch if n in ATTACHED]
to_fetch = [n for n in to_fetch if n not in ATTACHED]
for name in skipped:
    print(f"{name}: using attached input {ATTACHED[name]} (not downloading)")

print(f"fetching: {to_fetch or 'nothing (using your own data)'}\n")
for name in to_fetch:
    try:
        prepare(name, DATASET_ROOT)
    except Exception as error:
        print(f"{name} FAILED: {error}")

verify(DATASET_ROOT)

### Attached datasets — what was actually found

Community Scene Flow mirrors on Kaggle do not agree on a directory layout, so the loader
**searches** the attached path for one it recognises rather than assuming a fixed structure:

* **official** — `frames_finalpass/TRAIN/<A|B|C>/<scene>/left/*.png` + `disparity/TRAIN/...`
* **subset** — `train/image_clean/left/*.png` + `train/disparity/left/*.pfm` (the smaller
  `FlyingThings3D_subset` release, flat, lowercase split names)

Either can be nested any number of folders deep inside the attached dataset. This cell prints
what it resolved, so if a mirror is not recognised you can see its actual tree instead of
getting an opaque failure.

A mirror with **images but no disparity is still fully usable for training** — training here
is label-free. Disparity is only needed for Stage 4 benchmarking.

In [ ]:
if not ATTACHED:
    print("Nothing in ATTACHED. To use a Kaggle mirror instead of downloading, e.g.:")
    print('    ATTACHED = {"sceneflow": "/kaggle/input/sceneflow"}')
    print("\nAttached inputs currently visible:")
    for entry in sorted(glob.glob("/kaggle/input/*")):
        print("   ", entry)
else:
    from stereo.data.sceneflow import describe_tree, discover_sceneflow

    for name, path in ATTACHED.items():
        print("=" * 74)
        print(f"{name}  ->  {path}")
        print("=" * 74)
        if not os.path.isdir(path):
            print(f"  MISSING. Check the exact path: Add Data, then look under /kaggle/input.")
            for entry in sorted(glob.glob("/kaggle/input/*")):
                print("   attached input:", entry)
            continue

        if name == "sceneflow":
            for split in ("TRAIN", "TEST"):
                try:
                    layout = discover_sceneflow(path, split)
                    print(f"\n[{split}]")
                    print("  " + layout.describe().replace("\n", "\n  "))
                except FileNotFoundError as error:
                    print(f"\n[{split}] not found: {str(error).splitlines()[0]}")
            print("\nTree of what is actually attached:")
            print(describe_tree(path))
        else:
            print(describe_tree(path))

In [ ]:
# Does the model's search range actually cover this data?
#
# Middlebury and ETH3D ship an "ndisp" hint in calib.txt. That is CAMERA METADATA
# packaged with the images -- a search-range hint, not a per-pixel label -- and
# disp0GT.pfm is never opened here. Choosing a disparity range per dataset is a
# configuration decision the paper also makes by hand (256 / 384 / 512).
import cv2
from stereo.data.io import read_middlebury_calib

for name in ACTIVE_DATASETS:
    root = dataset_root_for(name)
    hints = []
    for calib in sorted(glob.glob(os.path.join(root, "*", "calib.txt")))[:64]:
        value = read_middlebury_calib(calib).get("ndisp")
        if value:
            hints.append(value)
    if not hints:
        continue
    # ndisp is quoted at the dataset's own resolution, so rescale to ours.
    sample_image = sorted(glob.glob(os.path.join(root, "*", "im0.png")))[:1]
    native_width = cv2.imread(sample_image[0]).shape[1] if sample_image else TRAIN_WIDTH
    needed = max(hints) * (TRAIN_WIDTH / native_width)
    verdict = "OK" if needed <= MAX_DISPARITY_PX else "TOO SMALL"
    print(f"{name:12s} native width {native_width:5d}, max ndisp hint {max(hints):.0f} "
          f"-> needs ~{needed:.0f} px at {TRAIN_WIDTH} wide; model reaches "
          f"{MAX_DISPARITY_PX} px  [{verdict}]")
    if needed > MAX_DISPARITY_PX:
        print(f"{'':12s}   raise MAX_DISPARITY_CAP (the paper uses 512 for Middlebury), "
              f"or lower TRAIN_WIDTH.")

## 6. Look at the training pairs

Images only — no ground truth is loaded here, in any mode. The assertion below proves it.

In [ ]:
import matplotlib.pyplot as plt
from stereo.data import DatasetMode, build_dataset
from stereo.data.augmentation import PhotometricAugmentConfig, ResizeConfig, build_train_transform
from stereo.data.registry import DatasetSpec

transform = build_train_transform(ResizeConfig(TRAIN_HEIGHT, TRAIN_WIDTH),
                                  PhotometricAugmentConfig(enabled=True, probability=1.0), seed=0)

# Preview whatever the Control Panel actually selected -- no hard-coded dataset.
if MODE == "adapt_unlabeled":
    preview_spec = DatasetSpec(type="folder", root=CUSTOM_DATA)
else:
    first = next(iter(ACTIVE_DATASETS), None)
    if first is None:
        raise RuntimeError("no dataset selected; set a non-zero weight in DATASETS")
    dataset_type, options = DATASET_TYPES[first]
    preview_spec = DatasetSpec(type=dataset_type, root=dataset_root_for(first), options=options)

preview = build_dataset(preview_spec, DatasetMode.TRAIN, transform)
print(f"{preview_spec.type} @ {preview_spec.root}")
print(f"{len(preview)} pairs | sample keys: {sorted(preview[0])}")
assert not any(k in preview[0] for k in ("disparity_gt", "depth_gt", "valid_gt_mask"))
print("confirmed: the training sample carries no ground truth")

rows = min(3, len(preview))
fig, axes = plt.subplots(rows, 2, figsize=(13, 3 * rows), squeeze=False)
for row in range(rows):
    sample = preview[row]
    for col, view in enumerate(("left", "right")):
        axes[row][col].imshow(sample[view].permute(1, 2, 0).numpy())
        axes[row][col].set_title(f"{view} (augmented) - {sample['metadata']['sample_id']}")
        axes[row][col].axis("off")
plt.tight_layout(); plt.show()

## 7. Assemble the training configuration

Pure translation of the Control Panel into the repo's `Config` object. Nothing new to decide
here — it is printed so you can see exactly what was built, and saved next to the checkpoints
so the run is reproducible from the CLI.

In [ ]:
from stereo.config import Config, LossWeights, TeacherConfig, config_to_yaml
from stereo.data.augmentation import GeometricAugmentConfig
from stereo.losses.pseudo_label import PseudoLabelFilterConfig
from stereo.model import StereoNetConfig

config = Config()
config.dynamic_disparity = False     # already resolved in the preflight
config.model = StereoNetConfig.for_width(TRAIN_WIDTH, downsample=DOWNSAMPLE,
                                         max_disparities_cap=MAX_DISPARITY_CAP)

config.data.resize = ResizeConfig(TRAIN_HEIGHT, TRAIN_WIDTH)
config.data.photometric_augmentation = PhotometricAugmentConfig(enabled=True)
config.data.geometric_augmentation = GeometricAugmentConfig(enabled=True, scale=(0.8, 1.2),
                                                            aspect=(0.9, 1.1))

# Training mixture, straight from the DATASETS dict.
if MODE == "adapt_unlabeled":
    config.data.train = [DatasetSpec(type="folder", root=CUSTOM_DATA, weight=1.0)]
    config.training.init_checkpoint = RESOLVED_INIT
    config.optimizer.learning_rate = LEARNING_RATE * ADAPT["lr_scale"]
    config.teacher.start_epoch = ADAPT["teacher_start_epoch"]
else:
    config.data.train = [
        DatasetSpec(type=DATASET_TYPES[name][0], root=dataset_root_for(name),
                    weight=weight, options=DATASET_TYPES[name][1])
        for name, weight in ACTIVE_DATASETS.items()
    ]
    config.optimizer.learning_rate = LEARNING_RATE
    config.teacher.start_epoch = TEACHER_START_EPOCH

config.data.validation = list(config.data.train)      # label-free validation

config.loss = LossWeights(photometric=W_PHOTOMETRIC, smoothness=W_SMOOTHNESS,
                          left_right=W_LEFT_RIGHT, pseudo=W_PSEUDO,
                          confidence=W_CONFIDENCE, low_resolution=W_LOW_RESOLUTION)

config.teacher.enabled = TEACHER_ENABLED
config.teacher.ema_decay = TEACHER_EMA_DECAY
config.teacher.ramp_epochs = TEACHER_RAMP_EPOCHS
config.teacher.filter = PseudoLabelFilterConfig(confidence_threshold=FILTER_CONFIDENCE,
                                                lr_threshold=FILTER_LR_PIXELS,
                                                photometric_threshold=FILTER_PHOTOMETRIC)

config.training.epochs = EPOCHS
config.training.batch_size = BATCH_SIZE
config.training.num_workers = NUM_WORKERS
config.training.use_amp = torch.cuda.is_available()
config.training.output_dir = OUTPUT_DIR
config.training.selection_metric = "val/photometric"   # LABEL-FREE checkpoint selection

os.makedirs(OUTPUT_DIR, exist_ok=True)
open(f"{OUTPUT_DIR}/config.yaml", "w").write(config_to_yaml(config))

print(f"num_disparities  : {config.model.num_disparities}")
print(f"training sets    : {[(s.type, s.weight) for s in config.data.train]}")
print(f"init checkpoint  : {config.training.init_checkpoint or 'none (random init)'}")
print(f"learning rate    : {config.optimizer.learning_rate:.1e}")
print(f"saved config     : {OUTPUT_DIR}/config.yaml")

## 8. Build the model

Random initialisation — no ImageNet weights, no pretrained stereo weights, no pretrained
Monodepth weights.

In [ ]:
from stereo.model import StereoNet

model = StereoNet(config.model)
print(f"parameters      : {model.num_parameters():,}")
print(f"cost volume     : {model.num_disparities_small} levels at 1/{model.scale}")
print(f"search range    : 0 .. {model.max_disparity} px at full resolution")
print(f"size divisor    : {model.size_divisor} (inputs are padded right/bottom, then cropped back)")

# Arbitrary resolution, including sizes that need padding.
model.eval()
for height, width in [(TRAIN_HEIGHT, TRAIN_WIDTH), (375, 1242), (540, 960), (224, 224)]:
    with torch.no_grad():
        out = model.forward_left(torch.rand(1, 3, height, width), torch.rand(1, 3, height, width))
    print(f"  {width}x{height} -> disparity {tuple(out['disparity'].shape)}")

## 9. Train, label-free

Stage 1 is photometric + smoothness + left-right consistency from random init.
Stage 2 adds the EMA teacher at `TEACHER_START_EPOCH`, ramped in.

**Watch `pseudo_cov`** once the teacher starts: near 0 means the reliability filter is
rejecting everything (loosen `FILTER_*`), near 1 means it is copying the teacher's errors
wholesale (tighten them). The trainer warns in both cases.

In [ ]:
if MODE == "evaluate":
    print("MODE is 'evaluate'; skipping training.")
else:
    from stereo.training import Trainer

    trainer = Trainer(config)
    best_checkpoint = trainer.fit()
    print("best label-free checkpoint:", best_checkpoint)

## 10. Label-free validation curves

No ground-truth metric is plotted — these are the quantities that actually selected the
checkpoint.

In [ ]:
import json

history_path = f"{OUTPUT_DIR}/history.json"
if os.path.exists(history_path):
    history = json.load(open(history_path))
    panels = [("train/total", "total loss"), ("train/photometric", "photometric"),
              ("train/left_right", "left-right consistency"), ("train/smoothness", "smoothness"),
              ("train/pseudo_valid_ratio", "pseudo-label coverage"),
              ("train/mean_confidence", "mean confidence")]
    fig, axes = plt.subplots(2, 3, figsize=(16, 7))
    for axis, (key, title) in zip(axes.flat, panels):
        values = [record.get(key) for record in history]
        if any(v is not None for v in values):
            axis.plot([r["epoch"] for r in history], values, marker="o", ms=3)
        if key == "train/photometric" and "val/photometric" in history[0]:
            axis.plot([r["epoch"] for r in history], [r["val/photometric"] for r in history],
                      marker="s", ms=3, label="validation")
            axis.legend()
        axis.set_title(title); axis.set_xlabel("epoch"); axis.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print("no history yet")

## 11. Qualitative check — still no ground truth

Predicted disparity, confidence, and the photometric reconstruction the model was actually
trained on.

In [ ]:
from stereo.data import collate_samples
from stereo.geometry import warp_right_to_left
from stereo.utils.visualization import colorize, to_numpy_image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device).eval()

batch = collate_samples([preview[i] for i in range(min(2, len(preview)))])
left, right = batch["left_clean"].to(device), batch["right_clean"].to(device)
with torch.no_grad():
    outputs = model(left, right, directions=("left", "right"))

reconstruction, valid = warp_right_to_left(right, outputs["left"]["disparity"])
residual = (reconstruction - left).abs().mean(dim=1, keepdim=True) * valid

rows = left.shape[0]
fig, axes = plt.subplots(rows, 3, figsize=(16, 3.5 * rows), squeeze=False)
for row in range(rows):
    for axis, (image, title) in zip(axes[row], [
            (to_numpy_image(left[row:row + 1]), "left"),
            (colorize(outputs["left"]["disparity"][row]), "predicted disparity"),
            (colorize(outputs["left"]["confidence"][row], 0, 1, cmap="viridis"),
             "confidence = exp(matchability)")]):
        axis.imshow(image); axis.set_title(title); axis.axis("off")
plt.tight_layout(); plt.show()
print(f"photometric residual (the training signal): {float(residual.sum() / valid.sum()):.4f}")

---
# 12. GROUND-TRUTH EVALUATION

**Everything below reads ground truth and runs only when `MODE == "evaluate"`.**

The checkpoint is loaded, frozen, and measured. No optimiser is constructed, and the
checkpoint being measured was selected by a label-free criterion, so ground truth did not
influence which weights got here either.

In [ ]:
if MODE != "evaluate":
    print(f"MODE is '{MODE}'. Ground truth is NOT loaded.")
    print("Set MODE = 'evaluate' in the Control Panel and re-run from the top.")
else:
    from stereo.evaluation import (PUBLISHED_RESULTS, evaluate_checkpoint, format_summary,
                                   get_protocol, write_results)
    from stereo.postprocess import PostProcessConfig
    from stereo.utils.checkpoint import build_model_from_checkpoint, checkpoint_hash

    frozen = build_model_from_checkpoint(RESOLVED_CHECKPOINT, map_location=device)
    protocol = get_protocol(PROTOCOL)
    print(f"checkpoint : {RESOLVED_CHECKPOINT}")
    print(f"sha256     : {checkpoint_hash(RESOLVED_CHECKPOINT)[:16]}...")
    print(f"protocol   : {protocol.name}")
    print(f"data       : {EVAL_ROOT}")
    print(f"notes      : {protocol.notes}")

### 12a. The paper's protocol: raw output

The paper's tables use *only the raw output of the learned model*, so post-processing is off.

In [ ]:
if MODE == "evaluate":
    summary_raw = evaluate_checkpoint(frozen, protocol, EVAL_ROOT, device,
                                      PostProcessConfig(enabled=False),
                                      max_samples=EVAL_MAX_SAMPLES)
    summary_raw["checkpoint"] = RESOLVED_CHECKPOINT
    write_results(summary_raw, f"{OUTPUT_DIR}/evaluation/{PROTOCOL}/raw")
    print(format_summary(summary_raw))

### 12b. With matchability post-processing

`exp(matchability) >= POSTPROCESS_CONFIDENCE` and a minimum depth-region of
`POSTPROCESS_MIN_REGION` px, as in the paper's on-robot pipeline. Reported separately, with
the pixel coverage stated, so the effect of the model and the effect of the filter stay
distinguishable.

In [ ]:
if MODE == "evaluate":
    summary_pp = evaluate_checkpoint(
        frozen, protocol, EVAL_ROOT, device,
        PostProcessConfig(enabled=True, confidence_threshold=POSTPROCESS_CONFIDENCE,
                          min_region_pixels=POSTPROCESS_MIN_REGION),
        max_samples=EVAL_MAX_SAMPLES)
    summary_pp["checkpoint"] = RESOLVED_CHECKPOINT
    write_results(summary_pp, f"{OUTPUT_DIR}/evaluation/{PROTOCOL}/postprocessed")
    print(format_summary(summary_pp))

### 12c. Comparison table

Every row is labelled *published*, *measured* or *not available*. Nothing is invented, and
rows measured under different protocols are never presented as equivalent.

In [ ]:
if MODE == "evaluate":
    def get(summary, key, variant="all"):
        # .get on the variant too: a protocol without a nonocc mask has no "nonocc" row.
        value = summary["disparity_metrics"].get(variant, {}).get(key)
        return f"{value:.3f}" if isinstance(value, (int, float)) else "n/a"

    rows = []
    if PROTOCOL == "sceneflow":
        published = PUBLISHED_RESULTS["sceneflow"]["metrics"]
        rows.append(("Original paper (Table IV)", "supervised, GT disparity",
                     f"{published['global_epe']:.3f}", f"{published['global_bad_1']:.1f}",
                     "PUBLISHED"))
        rows.append(("Reference mmstereo checkpoint", "supervised", "n/a", "n/a",
                     "NOT AVAILABLE (no released weights)"))
        rows.append(("This implementation (raw)", "label-free self-supervised",
                     get(summary_raw, "global_epe"), get(summary_raw, "global_bad_1"), "MEASURED"))
        rows.append(("This implementation (post-processed)", "label-free self-supervised",
                     get(summary_pp, "global_epe"), get(summary_pp, "global_bad_1"),
                     f"MEASURED on {summary_pp['ground_truth_pixel_coverage'] * 100:.0f}% of pixels"))
        header = ("Model", "Training", "EPE (px)", "%Bad(1.0)", "Status")
    elif PROTOCOL == "middlebury2014":
        published = PUBLISHED_RESULTS["middlebury2014_test"]["metrics"]
        rows.append(("Original paper (Table V, TEST split)", "supervised, GT disparity",
                     f"{published['image_bad_2_nonocc']}/{published['image_bad_2_all']}",
                     f"{published['image_avgerr_nonocc']}/{published['image_avgerr_all']}",
                     "PUBLISHED - DIFFERENT SPLIT"))
        rows.append(("This implementation (raw, TRAINING split)", "label-free self-supervised",
                     f"{get(summary_raw, 'image_bad_2', 'nonocc')}/{get(summary_raw, 'image_bad_2')}",
                     f"{get(summary_raw, 'image_avgerr', 'nonocc')}/{get(summary_raw, 'image_avgerr')}",
                     "MEASURED"))
        header = ("Model", "Training", "bad2.0 nocc/all", "avgerr nocc/all", "Status")
    else:
        rows.append((f"This implementation (raw)", "label-free self-supervised",
                     get(summary_raw, "global_epe"), get(summary_raw, "global_bad_1"), "MEASURED"))
        rows.append((f"This implementation (post-processed)", "label-free self-supervised",
                     get(summary_pp, "global_epe"), get(summary_pp, "global_bad_1"), "MEASURED"))
        header = ("Model", "Training", "EPE (px)", "%Bad(1.0)", "Status")
        print(f"NOTE: the paper reports no accuracy numbers for {PROTOCOL}, so there is no "
              "published row to compare against.\n")

    widths = [max(len(str(row[i])) for row in [header] + rows) for i in range(len(header))]
    line = lambda row: "  ".join(str(cell).ljust(widths[i]) for i, cell in enumerate(row))
    print(f"Dataset  : {protocol.dataset_type}   Split: {protocol.split}")
    print(f"Protocol : {protocol.primary_source}")
    print(f"Mask     : {protocol.max_disparity_source}")
    print(f"Scaling  : {'median' if protocol.median_scaling else 'none (metric prediction)'}\n")
    print(line(header)); print("-" * (sum(widths) + 2 * len(widths)))
    for row in rows: print(line(row))
    if PROTOCOL == "middlebury2014":
        print(f"\nCAVEAT: {PUBLISHED_RESULTS['middlebury2014_test']['caveat']}")

### 12d. Error visualisation

Sparse ground truth is never densified for display — invalid pixels stay grey.

In [ ]:
if MODE == "evaluate":
    from stereo.data import build_benchmark_dataset
    from stereo.evaluation.disparity_metrics import disparity_valid_mask
    from stereo.postprocess import upsample_confidence

    dataset = build_benchmark_dataset(DatasetSpec(type=protocol.dataset_type, root=EVAL_ROOT,
                                                  options=dict(protocol.dataset_options)))
    for index in range(min(2, len(dataset))):
        sample = collate_samples([dataset[index]])
        left_image, right_image = sample["left"].to(device), sample["right"].to(device)
        with torch.no_grad():
            output = frozen.forward_left(left_image, right_image)

        disparity = output["disparity"]
        disparity_gt = sample["disparity_gt"].to(device)
        valid = disparity_valid_mask(disparity_gt, protocol.max_disparity, protocol.min_disparity,
                                     sample["valid_gt_mask"].to(device))
        error = (disparity - disparity_gt).abs()
        vmax = float(disparity_gt[valid].max()) if valid.any() else None

        panels = [(to_numpy_image(left_image), "left image"),
                  (colorize(disparity, 0, vmax), "predicted disparity"),
                  (colorize(disparity_gt, 0, vmax, mask=valid), "ground-truth disparity"),
                  (colorize(error, 0, 5, mask=valid, cmap="inferno"), "|error| (px), 0-5"),
                  (colorize(valid.float(), 0, 1, cmap="gray"), "valid GT mask"),
                  (colorize(upsample_confidence(output["confidence"], disparity.shape[-2:]), 0, 1,
                            cmap="viridis"), "confidence")]
        fig, axes = plt.subplots(2, 3, figsize=(17, 7))
        for axis, (image, title) in zip(axes.flat, panels):
            axis.imshow(image); axis.set_title(title, fontsize=10); axis.axis("off")
        epe = float(error[valid].mean()) if valid.any() else float("nan")
        fig.suptitle(f"{sample['metadata']['sample_id'][0]} - EPE {epe:.3f} px")
        plt.tight_layout(); plt.show()

## 13. Label-leakage audit

Runs in every mode. If this fails, nothing else in the notebook means anything.

In [ ]:
subprocess.run([sys.executable, "scripts/audit_label_leakage.py", "--strict"], check=True)

## 14. Save artefacts

Kaggle keeps `/kaggle/working`. Checkpoints and evaluation output are already there; this
just lists what a rerun would pick up.

In [ ]:
for directory, _, filenames in os.walk("/kaggle/working/outputs"):
    for filename in sorted(filenames):
        path = os.path.join(directory, filename)
        print(f"{os.path.getsize(path) / 1e6:8.2f} MB  {path}")